### Ce notebook permet de faire une vérification de la qualité des données avant toute consolidation, visualisation ou modélisation.

In [1]:
import pandas as pd
from pathlib import Path

###  Chargement de données

In [2]:
# Chemin d'accès aux données brutes

RAW_DATA_DIR = Path("../..") / "data" / "raw"

# Charger les datasets

files = {
    "idmc": "data_idmc_depuis_2000.csv",
    "solutions": "data_solutions_depuis_2000.csv",
    "decisions": "decisions_asile_depuis_2000.csv",
    "demandes": "demandes_asile_depuis_2000.csv",
    "demographie": "demographie_depuis_2000.csv",
    "pays": "countries.csv"
}


dfs = {
    name: pd.read_csv(RAW_DATA_DIR / filename)
    for name, filename in files.items()
}

for name, df in dfs.items():
    print(f"{name:15} : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

idmc            : 934 lignes × 10 colonnes
solutions       : 21,258 lignes × 13 colonnes
decisions       : 113,929 lignes × 17 colonnes
demandes        : 120,597 lignes × 14 colonnes
demographie     : 116,781 lignes × 24 colonnes
pays            : 232 lignes × 16 colonnes


### Cohérence des décisions

In [3]:
df = dfs["decisions"].copy()

decision_cols = [
    "dec_recognized",
    "dec_other",
    "dec_rejected",
    "dec_closed",
    "dec_total"
]

for col in decision_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["dec_sum"] = (
    df[
        [
            "dec_recognized",
            "dec_other",
            "dec_rejected",
            "dec_closed"
        ]
    ]
    .sum(axis=1, min_count=1)
)

df["dec_ecart"] = df["dec_total"] - df["dec_sum"]

anomalies_decisions = df[
    df["dec_ecart"].fillna(0) != 0
]

print("Nombre d'écarts :", len(anomalies_decisions))

display(
    anomalies_decisions[
        [
            "year",
            "coo_iso",
            "coa_iso",
            "dec_total",
            "dec_sum",
            "dec_ecart"
        ]
    ].head(20)
)

Nombre d'écarts : 444


,year,coo_iso,coa_iso,dec_total,dec_sum,dec_ecart
73,2000,ARM,CZE,43,41,2
77,2000,BGR,CZE,89,90,-1
78,2000,COG,CZE,14,15,-1
100,2000,UKR,CZE,64,63,1
201,2001,UNK,MLT,5,0,5
347,2002,RUS,GBR,1815,1810,5
360,2002,UNK,MLT,185,0,185
742,2006,ALB,GBR,85,90,-5
743,2006,DZA,GBR,160,155,5
744,2006,AGO,GBR,90,85,5
